# TARGETED WIENER RERUN

**File**: `03_mvtec_patchcore_aug_wiener.ipynb`

This notebook is a surgically stripped-down version of the original pipeline.
It is designed solely to regenerate the misspecified Wiener deconvolution rows
using the corrected, per-severity PSF parameters.

**It ONLY executes:**
- `gaussian_blur` and `motion_blur`
- `mild` and `moderate` severities
- Wiener deconvolution rescue

All other corruptions, severities, and rescue methods have been removed for speed.

In [1]:
# ---------------------------------------------------------------------------
# 2. Embedded Corruption & Rescue Functions (Stripped down for Wiener rerun)
# ---------------------------------------------------------------------------

def apply_gaussian_blur(image, sigma, kernel_size):
    t = A.GaussianBlur(blur_limit=(kernel_size, kernel_size), sigma_limit=(sigma, sigma), p=1.0)
    return t(image=image)["image"]

def apply_motion_blur(image, kernel_size):
    t = A.MotionBlur(blur_limit=(kernel_size, kernel_size), p=1.0)
    return t(image=image)["image"]

def apply_corruption(image, ctype, severity, config, seed=42):
    params = config["corruptions"][ctype][severity]
    if ctype == "gaussian_blur": return apply_gaussian_blur(image, params["sigma"], params["kernel_size"])
    elif ctype == "motion_blur": return apply_motion_blur(image, params["kernel_size"])
    else: raise ValueError(f"Unknown corruption type: {ctype}")

def apply_wiener_deconv(image: np.ndarray, sigma: float, kernel_size: int, balance: float = 0.1) -> np.ndarray:
    import cv2, numpy as np
    from skimage.restoration import wiener
    psf = np.zeros((kernel_size, kernel_size))
    center = kernel_size // 2
    psf[center, center] = 1.0
    psf = cv2.GaussianBlur(psf, (kernel_size, kernel_size), sigmaX=sigma, sigmaY=sigma)
    psf /= psf.sum()
    out = np.zeros_like(image, dtype=np.float64)
    for i in range(3):
        out[:, :, i] = wiener(image[:, :, i] / 255.0, psf, balance, clip=False)
    out = np.clip(out * 255, 0, 255).astype(np.uint8)
    return out

def apply_motion_wiener_deconv(image: np.ndarray, kernel_size: int, balance: float = 0.1) -> np.ndarray:
    import cv2, numpy as np
    from skimage.restoration import wiener
    psf = np.zeros((kernel_size, kernel_size))
    psf[kernel_size // 2, :] = 1.0 / kernel_size
    angle = 0
    M = cv2.getRotationMatrix2D((kernel_size / 2, kernel_size / 2), angle, 1)
    psf = cv2.warpAffine(psf, M, (kernel_size, kernel_size))
    psf /= psf.sum()
    out = np.zeros_like(image, dtype=np.float64)
    for i in range(3):
        out[:, :, i] = wiener(image[:, :, i] / 255.0, psf, balance, clip=False)
    out = np.clip(out * 255, 0, 255).astype(np.uint8)
    return out

def get_rescue_map(severity, config):
    gauss_p = config["corruptions"]["gaussian_blur"][severity]
    motion_p = config["corruptions"]["motion_blur"][severity]
    return {
        "gaussian_blur": [("Wiener", lambda img, s=gauss_p["sigma"], k=gauss_p["kernel_size"]:
                           apply_wiener_deconv(img, sigma=s, kernel_size=k))],
        "motion_blur": [("Wiener (Motion PSF)", lambda img, k=motion_p["kernel_size"]:
                         apply_motion_wiener_deconv(img, kernel_size=k))]
    }


Script started at Sat Jun  6 05:55:54 2026
Graceful timeout set to 11.5 hours.
Installing missing dependency: anomalib...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 17.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


Installing missing dependency: scikit-image...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Pre-generating augmented training data (aug_prob=0.5)...
  ✅ bottle done
  ✅ cable done
  ✅ capsule done
  ✅ carpet done
  ✅ grid done
  ✅ hazelnut done
  ✅ leather done
  ✅ metal_nut done
  ✅ pill done
  ✅ screw done
  ✅ tile done
  ✅ toothbrush done
  ✅ transistor done
  ✅ wood done


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.45634350180625916    │
│       image_F1Score       │    0.8530465960502625     │
│        pixel_AUROC        │    0.8915958404541016     │
│       pixel_F1Score       │   0.016304627060890198    │
└───────────────────────────┴───────────────────────────┘

[PHASE 3] Rescue: fog_haze (severe) + Dehaze (Dark Channel)


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            0.5            │
│       image_F1Score       │    0.8530465960502625     │
│        pixel_AUROC        │    0.8451088666915894     │
│       pixel_F1Score       │   0.0052657583728432655   │
└───────────────────────────┴───────────────────────────┘

⏩ Skipping screw (Seed 456)
⏩ Skipping tile (Seed 42)
⏩ Skipping tile (Seed 123)
⏩ Skipping tile (Seed 456)
⏩ Skipping toothbrush (Seed 42)
⏩ Skipping toothbrush (Seed 123)
⏩ Skipping toothbrush (Seed 456)
⏩ Skipping transistor (Seed 42)
⏩ Skipping transistor (Seed 123)
⏩ Skipping transistor (Seed 456)
⏩ Skipping wood (Seed 42)
⏩ Skipping wood (Seed 123)
⏩ Skipping wood (Seed 456)
⏩ Skipping zipper (Seed 42)
⏩ Skipping zipper (Seed 123)
⏩ Skipping zipper (Seed 456)

EXPERIMENT COMPLETE
